In [ ]:
import time
import numpy as np
import pandas as pd
from geotessera.store import GeoTesseraZarr

STORE_URL = "https://s3.us-west-2.amazonaws.com/tessera-embeddings/v1/zarr/v2/store.zarr"
YEAR = 2022
BATCH_SIZE = 200   # points per batched call — tune up/down based on how it performs

In [ ]:
df = pd.read_csv("../data/processed/ukbms_sites_2022_ready.csv")
gt = GeoTesseraZarr(store_url=STORE_URL)

In [5]:
sample_points = list(zip(df['lon'][:20], df['lat'][:20]))
for year in [2020, 2021, 2022, 2023, 2024]:
    try:
        emb = gt.sample_embeddings_at_points(sample_points, year=year)
        n_valid = sum(1 for row in emb if row is not None)
        print(f"year {year}: {n_valid}/{len(sample_points)} valid")
    except Exception as e:
        print(f"year {year}: failed — {e}")

year 2020: 20/20 valid
year 2021: 20/20 valid
year 2022: 20/20 valid
year 2023: 20/20 valid
year 2024: 20/20 valid


In [ ]:
points = list(zip(df['lon'], df['lat']))
n = len(points)
embs = np.full((n, 128), np.nan, dtype=np.float32)

t0 = time.time()
for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch = points[start:end]
    try:
        # ONE network round-trip for the whole batch, instead of 200 separate ones
        result = gt.sample_points(batch, year=YEAR)
        embs[start:end] = result
    except Exception as e:
        print(f"batch [{start}:{end}] failed: {e}", flush=True)

    elapsed = time.time() - t0
    rate = elapsed / end
    eta_min = rate * (n - end) / 60
    print(f"{end}/{n} done, {elapsed:.0f}s elapsed, ~{eta_min:.1f} min remaining", flush=True)

emb_df = pd.DataFrame(embs, columns=[f'emb_{i}' for i in range(128)])
df_full = pd.concat([df.reset_index(drop=True), emb_df], axis=1)
valid = ~np.isnan(df_full['emb_0'])
df_full = df_full[valid]
print(f"{len(df_full)} of {n} sites have valid embeddings", flush=True)
df_full.to_csv("../data/processed/ukbms_sites_with_embeddings.csv", index=False)